# K-Means Clustering for Grafana Logs

This notebook performs K-Means clustering on the extracted features from Grafana logs.

## K-Means Algorithm:
- **Type**: Partitioning-based clustering
- **Advantages**: Fast, scalable, works well with spherical clusters
- **Disadvantages**: Requires predefined k, sensitive to initialization, assumes equal-sized clusters

## Steps:
1. Load feature matrices
2. Determine optimal number of clusters (Elbow method, Silhouette analysis)
3. Perform K-Means clustering
4. Evaluate clustering quality (Silhouette score, Davies-Bouldin index, Calinski-Harabasz score)
5. Visualize clusters
6. Analyze cluster characteristics
7. Benchmark performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
from scipy.spatial.distance import cdist
import time
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## 1. Load Feature Matrices

In [ ]:
# Load scaled features
X_scaled = pd.read_csv('features_scaled.csv')
print(f"Scaled features shape: {X_scaled.shape}")

# Load PCA features
X_pca = pd.read_csv('features_pca.csv')
print(f"PCA features shape: {X_pca.shape}")

# Load metadata
metadata = pd.read_csv('metadata.csv')
print(f"Metadata shape: {metadata.shape}")

# Convert to numpy arrays
X_scaled_array = X_scaled.values
X_pca_array = X_pca.values

print("\nData loaded successfully!")

## 2. Determine Optimal Number of Clusters

### 2.1 Elbow Method

In [ ]:
# Test different numbers of clusters
K_range = range(2, 21)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
calinski_harabasz_scores = []

print("Testing different numbers of clusters...")
print("This may take a few minutes...\n")

for k in K_range:
    print(f"Testing k={k}...", end=' ')
    
    # Fit K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled_array)
    
    # Calculate metrics
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled_array, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled_array, labels))
    calinski_harabasz_scores.append(calinski_harabasz_score(X_scaled_array, labels))
    
    print(f"Silhouette: {silhouette_scores[-1]:.4f}")

print("\nOptimization complete!")

In [ ]:
# Plot elbow curves
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Inertia (Elbow method)
axes[0, 0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
axes[0, 0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette Score
axes[0, 1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 1].set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
best_silhouette_k = K_range[np.argmax(silhouette_scores)]
axes[0, 1].axvline(x=best_silhouette_k, color='r', linestyle='--', label=f'Best k={best_silhouette_k}')
axes[0, 1].legend()

# Davies-Bouldin Index (lower is better)
axes[1, 0].plot(K_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 0].set_ylabel('Davies-Bouldin Index', fontsize=12)
axes[1, 0].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
best_db_k = K_range[np.argmin(davies_bouldin_scores)]
axes[1, 0].axvline(x=best_db_k, color='g', linestyle='--', label=f'Best k={best_db_k}')
axes[1, 0].legend()

# Calinski-Harabasz Score (higher is better)
axes[1, 1].plot(K_range, calinski_harabasz_scores, 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 1].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[1, 1].set_title('Calinski-Harabasz Index (Higher is Better)', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
best_ch_k = K_range[np.argmax(calinski_harabasz_scores)]
axes[1, 1].axvline(x=best_ch_k, color='r', linestyle='--', label=f'Best k={best_ch_k}')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('kmeans_optimal_k.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nOptimal k suggestions:")
print(f"  - Best Silhouette Score: k={best_silhouette_k} (score: {max(silhouette_scores):.4f})")
print(f"  - Best Davies-Bouldin Index: k={best_db_k} (score: {min(davies_bouldin_scores):.4f})")
print(f"  - Best Calinski-Harabasz Score: k={best_ch_k} (score: {max(calinski_harabasz_scores):.2f})")

### 2.2 Select Optimal k

In [ ]:
# Use the k with best silhouette score
optimal_k = best_silhouette_k
print(f"Selected optimal k: {optimal_k}")
print(f"\nYou can change this value if needed based on domain knowledge.")

## 3. Perform K-Means Clustering with Optimal k

In [ ]:
# Perform K-Means with optimal k
print(f"Performing K-Means clustering with k={optimal_k}...")

start_time = time.time()
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20, max_iter=500)
cluster_labels = kmeans_final.fit_predict(X_scaled_array)
clustering_time = time.time() - start_time

print(f"Clustering completed in {clustering_time:.2f} seconds")
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} samples ({count/len(cluster_labels)*100:.2f}%)")

## 4. Evaluate Clustering Quality

In [ ]:
# Calculate evaluation metrics
silhouette = silhouette_score(X_scaled_array, cluster_labels)
davies_bouldin = davies_bouldin_score(X_scaled_array, cluster_labels)
calinski_harabasz = calinski_harabasz_score(X_scaled_array, cluster_labels)
inertia = kmeans_final.inertia_

print("=" * 80)
print("K-MEANS CLUSTERING EVALUATION METRICS")
print("=" * 80)
print(f"\nNumber of clusters: {optimal_k}")
print(f"Number of samples: {len(cluster_labels):,}")
print(f"Clustering time: {clustering_time:.2f} seconds")
print(f"\nQuality Metrics:")
print(f"  - Silhouette Score: {silhouette:.4f} (range: [-1, 1], higher is better)")
print(f"  - Davies-Bouldin Index: {davies_bouldin:.4f} (lower is better)")
print(f"  - Calinski-Harabasz Score: {calinski_harabasz:.2f} (higher is better)")
print(f"  - Inertia: {inertia:.2f}")
print("\n" + "=" * 80)

# Store metrics for comparison
kmeans_metrics = {
    'algorithm': 'K-Means',
    'n_clusters': optimal_k,
    'silhouette_score': silhouette,
    'davies_bouldin_index': davies_bouldin,
    'calinski_harabasz_score': calinski_harabasz,
    'inertia': inertia,
    'clustering_time': clustering_time,
    'n_samples': len(cluster_labels)
}

## 5. Visualize Clusters

In [ ]:
# Visualize clusters using PCA (first 2 components)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PC1 vs PC2
scatter1 = axes[0].scatter(X_pca_array[:, 0], X_pca_array[:, 1], 
                           c=cluster_labels, cmap='viridis', alpha=0.6, s=30)
axes[0].set_xlabel('First Principal Component', fontsize=12)
axes[0].set_ylabel('Second Principal Component', fontsize=12)
axes[0].set_title('K-Means Clusters (PC1 vs PC2)', fontsize=14, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: PC2 vs PC3
scatter2 = axes[1].scatter(X_pca_array[:, 1], X_pca_array[:, 2], 
                           c=cluster_labels, cmap='viridis', alpha=0.6, s=30)
axes[1].set_xlabel('Second Principal Component', fontsize=12)
axes[1].set_ylabel('Third Principal Component', fontsize=12)
axes[1].set_title('K-Means Clusters (PC2 vs PC3)', fontsize=14, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig('kmeans_clusters_pca.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3D visualization
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(X_pca_array[:, 0], X_pca_array[:, 1], X_pca_array[:, 2],
                     c=cluster_labels, cmap='viridis', alpha=0.6, s=20)

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_zlabel('PC3', fontsize=12)
ax.set_title('K-Means Clusters in 3D PCA Space', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Cluster', shrink=0.6)

plt.tight_layout()
plt.savefig('kmeans_clusters_3d.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Cluster Characteristics Analysis

In [ ]:
# Add cluster labels to metadata
metadata['cluster'] = cluster_labels

# Analyze cluster characteristics
print("Cluster Characteristics:\n")
print("=" * 80)

for cluster_id in range(optimal_k):
    cluster_data = metadata[metadata['cluster'] == cluster_id]
    
    print(f"\nCluster {cluster_id} (n={len(cluster_data)}):")
    print("-" * 80)
    
    # Top panels
    print("  Top 5 Panels:")
    top_panels = cluster_data['panel_title'].value_counts().head(5)
    for panel, count in top_panels.items():
        print(f"    - {panel}: {count} ({count/len(cluster_data)*100:.1f}%)")
    
    # Top services
    print("\n  Top 5 Services:")
    top_services = cluster_data['service'].value_counts().head(5)
    for service, count in top_services.items():
        print(f"    - {service}: {count} ({count/len(cluster_data)*100:.1f}%)")
    
    # Value statistics
    print("\n  Value Statistics:")
    print(f"    - Mean: {cluster_data['value'].mean():.2f}")
    print(f"    - Median: {cluster_data['value'].median():.2f}")
    print(f"    - Std: {cluster_data['value'].std():.2f}")
    print(f"    - Min: {cluster_data['value'].min():.2f}")
    print(f"    - Max: {cluster_data['value'].max():.2f}")

print("\n" + "=" * 80)

In [ ]:
# Visualize cluster characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Cluster size distribution
cluster_sizes = metadata['cluster'].value_counts().sort_index()
axes[0, 0].bar(cluster_sizes.index, cluster_sizes.values, color='steelblue')
axes[0, 0].set_xlabel('Cluster ID', fontsize=12)
axes[0, 0].set_ylabel('Number of Samples', fontsize=12)
axes[0, 0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Value distribution by cluster
metadata.boxplot(column='value', by='cluster', ax=axes[0, 1])
axes[0, 1].set_xlabel('Cluster ID', fontsize=12)
axes[0, 1].set_ylabel('Value', fontsize=12)
axes[0, 1].set_title('Value Distribution by Cluster', fontsize=14, fontweight='bold')
plt.sca(axes[0, 1])
plt.xticks(rotation=0)

# Panel distribution across clusters
top_panels = metadata['panel_title'].value_counts().head(8).index
panel_cluster_counts = pd.crosstab(metadata['panel_title'], metadata['cluster'])
panel_cluster_counts.loc[top_panels].plot(kind='bar', stacked=True, ax=axes[1, 0], colormap='viridis')
axes[1, 0].set_xlabel('Panel Title', fontsize=12)
axes[1, 0].set_ylabel('Count', fontsize=12)
axes[1, 0].set_title('Top Panels Distribution Across Clusters', fontsize=14, fontweight='bold')
axes[1, 0].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.sca(axes[1, 0])
plt.xticks(rotation=45, ha='right')

# Service distribution across clusters
top_services = metadata['service'].value_counts().head(8).index
service_cluster_counts = pd.crosstab(metadata['service'], metadata['cluster'])
service_cluster_counts.loc[top_services].plot(kind='bar', stacked=True, ax=axes[1, 1], colormap='viridis')
axes[1, 1].set_xlabel('Service', fontsize=12)
axes[1, 1].set_ylabel('Count', fontsize=12)
axes[1, 1].set_title('Top Services Distribution Across Clusters', fontsize=14, fontweight='bold')
axes[1, 1].legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.sca(axes[1, 1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('kmeans_cluster_characteristics.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
# Save cluster assignments
results_df = metadata.copy()
results_df.to_csv('kmeans_cluster_assignments.csv', index=False)
print("Cluster assignments saved to: kmeans_cluster_assignments.csv")

# Save metrics
import json
with open('kmeans_metrics.json', 'w') as f:
    json.dump(kmeans_metrics, f, indent=2)
print("Metrics saved to: kmeans_metrics.json")

# Save model
import pickle
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans_final, f)
print("Model saved to: kmeans_model.pkl")

print("\nK-Means clustering complete!")